In [22]:
from langchain.agents.agent_toolkits import create_retriever_tool,create_conversational_retrieval_agent
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.agents.openai_functions_agent.agent_token_buffer_memory import AgentTokenBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


In [23]:
load_dotenv()


True

In [24]:
llm = ChatBedrock(
            model_id="anthropic.claude-3-5-sonnet-20240620-v1:0",
            model_kwargs={"temperature": 0},
            region_name="us-east-1"
        )

In [ ]:
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Optional, Type,List
from langchain.callbacks.manager import (
    CallbackManagerForToolRun,
)
import requests
import os

class SaveResarchInfoSchema(BaseModel):
    perspective: str = Field(description="The type of place your looking for EX:(park,restaraunt,gym)")
    objective: str = Field(description="The user's latitude")
    space:str = Field(description="The user's longitude")


class SaveResarchInfoTool(BaseTool):
    name = "SaveResarchInfo"
    description = "Finds a list of 10 nearby places with their name, address,and website_url"
    args_schema: Type[SearchNearbyPlacesSchema] = SearchNearbyPlacesSchema

    def _run(
        self,
        latitude,
        longitude,
        place_type,
        run_manager: Optional[CallbackManagerForToolRun] = None,
    ) -> str:
        """Use the tool."""
        url = "https://places.googleapis.com/v1/places:searchNearby"
        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": os.getenv('GOOGLE_MAPS_API_KEY'),
            "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.types,places.websiteUri"
        }

        data = {
            "includedTypes": [place_type],
            "maxResultCount": 10,
            "locationRestriction": {
                "circle": {
                    "center": {
                        "latitude": latitude,
                        "longitude": longitude
                    },
                    "radius": 500.0
                },
            },
            "rankPreference": "DISTANCE"
        }

        response = requests.post(url, json=data, headers=headers)

        indexed_places = []

        for place in response.json()['places']:
            indexed_place = {
                'types': place['types'],
                'address': place['formattedAddress'],
                'url': place['websiteUri'],
                'name': place['displayName']['text']
            }
        indexed_places.append(indexed_place)
        return indexed_places


    async def _arun(
        self,
        latitude,
        longitude,
        place_type,
    ) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("search does not support async")

In [25]:
memory_key = "history"
memory = AgentTokenBufferMemory(memory_key=memory_key, llm=llm)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
              You are an expert research assistant named Sherlock Holmie. Your job is to understand a user's research goals in order to 
""",
        ),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [26]:
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, memory=memory, verbose=True,
                                   return_intermediate_steps=True)

while True:
    user_input = input("Type your message (type 'exit' to quit): ")
    
    if user_input.lower() == 'exit':
        break  # exit the loop if the user types 'exit'
    
    result = agent_executor({"input": user_input})
    
    print("Agent Output:", result['output'])

NameError: name 'tools' is not defined